# Build a Knowledge Graph for Your IP Multicast (PIM Sparse-Mode) Design — the guided version

*A warm, step-by-step lab for network engineers. You bring the multicast. We bring the graph.*

Unicast is a private conversation: one source, one destination, one path. **Multicast is a
broadcast tree** — one source feeding *many* receivers down a branching distribution tree. And
in **PIM sparse-mode**, that tree has a meeting place the unicast world never worries about: a
**Rendezvous Point (RP)**. Every group's shared tree is *rooted* at an RP; sources register
there, receivers join toward it, and traffic flows down from it.

That makes the RP — and each branch of the tree — a **single point** whose failure is invisible
to almost every tool you own. Lose the RP for a group, or lose one branch, and *specific*
groups go dark while the entire rest of the network looks perfectly healthy. The IPTV wall
freezes, the market-data feed stalls, the imaging stream drops — and unicast `ping` says
everything is fine.

A **knowledge graph** makes that hidden tree explicit. By the end of this notebook you will have
built, from an empty page, a small graph that answers the question no `show ip mroute` can:

> *"rp-core-1 just went down. Which business service is now at risk?"*

and watched it print **`IPTV Channel Lineup`** — a fact that lives in nobody's mroute table.

### First, calm the nerves: this is **not** machine learning
No training. No model. No embeddings. No AI guessing in the dark. A knowledge graph is just
**structured facts** (things, and the named links between them) plus **queries** that walk those
links. Everything is **deterministic and inspectable** — run it twice, get the same answer, and
you can point at every node that produced it. If you can read an mroute table, you can read
every line in this lab.

### What you need
Nothing. This runs in **Google Colab** with zero setup, using plain **Python + networkx +
matplotlib** (both already installed in Colab). No database, no Docker, no credentials. Press
*Runtime → Run all*, or run each cell in order with **Shift+Enter**.


## The whole vocabulary — five words, each one a multicast thing you already know

Before any code, here is the *entire* mental model. Everything after this is these five ideas,
repeated. You don't have to learn what any of them *mean* — only which multicast thing each one
maps onto.

| Graph word | Plain meaning | The multicast thing it already is |
|---|---|---|
| **node** | a thing | a source, a group, an RP, a router, a service |
| **edge** | a named, directed link between two nodes | "this source sends to that group", a tree branch |
| **label** | a node's *type* | `Source`, `Group`, `RP`, `Router`, `Service` |
| **property** | a fact stored *on* a node or edge | `state='down'`, `group_address='239.1.1.1'` |
| **traversal** | walking edges to answer a question | "which services ride a group whose RP just died?" |

That's it. Hold those five words and the rest is just your day job wearing a new hat.

One idea deserves a spotlight, because it is the whole trick: **a fact can live on an edge, not
just a node.** "This branch of the tree is down" is not really a property of the router above it
or below it — it is a property of the *branch between them*. PIM already knows this in its bones
(the interface either forwards the group or it doesn't). In a graph you write it down literally:
the failure lives **on the edge**. Keep an eye out for that moment — it is where a topology
diagram turns into something you can *query*.


## Setup — one import, one empty graph

**Theory (10 seconds).** `networkx` is a tiny Python library for building graphs in memory. We
will use a **`DiGraph`** — a *directed* graph, where every edge has a direction, an arrow. That
matters to us: multicast is *relentlessly* directional. A **source sends to** a group; a group
is **rooted at** an RP; traffic **forwards down** the tree from the RP toward receivers — never
the other way. Directed edges are the honest choice.

Run the next cell. It imports the tools and hands you an empty graph to fill.


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Both libraries ship pre-installed in Colab — nothing to pip install.
# A DiGraph is a *directed* graph: every edge is an arrow, from one node to another.
G = nx.DiGraph()

print('Empty graph ready:', G)
print('Nodes:', G.number_of_nodes(), ' Edges:', G.number_of_edges())

**Checkpoint.** You should see `Empty graph ready` and `Nodes: 0  Edges: 0`. That empty `G` is
your blank whiteboard. Every step from here just adds nodes and edges to it.


## Step 1 — Sources and Groups: the multicast actors, as nodes

**Theory.** A multicast **group** is an address — something in `224.0.0.0/4` — that stands for
a *stream*, not a host. `239.1.1.1` isn't a box you can `ping`; it's "the IPTV channel feed,"
an address many receivers agree to listen to. A **source** is the one box actually transmitting
into that group. That source-to-group relationship is the first thing worth writing down.

In graph terms, each source and each group becomes a **node**. The source gets a `Source`
**label** and an `address`; the group gets a `Group` **label**, its `group_address`, and the
`app` it carries. We connect them with a `SENDS_TO` **edge**. We'll model two real streams from
the field:

- **`239.1.1.1`** — an **IPTV** channel feed, sourced by the head-end encoder.
- **`239.2.2.2`** — a **market-data** feed, sourced by the exchange gateway.

Notice we are *not* dumping the mroute table in here. A node per group and per source — not a
row per (S,G) forwarding entry. We store the *shape* of the design, and we'll come back to that
principle by name later.


In [ ]:
# add_node(name, **properties). The name is a unique handle; label & address are facts on it.
G.add_node('iptv-headend', label='Source', address='10.10.0.5')
G.add_node('market-feed',  label='Source', address='10.20.0.5')

G.add_node('239.1.1.1', label='Group', group_address='239.1.1.1', app='IPTV')
G.add_node('239.2.2.2', label='Group', group_address='239.2.2.2', app='market-data')

# SENDS_TO: a source transmits into a group.
G.add_edge('iptv-headend', '239.1.1.1', rel='SENDS_TO')
G.add_edge('market-feed',  '239.2.2.2', rel='SENDS_TO')

print(G.number_of_nodes(), 'nodes so far:', list(G.nodes))
print('Facts on the IPTV group:', G.nodes['239.1.1.1'])

**Checkpoint.** Four nodes — two sources, two groups — and the facts on `239.1.1.1` show
`group_address='239.1.1.1'` and `app='IPTV'`. Each source now points at the group it feeds. You
just wrote your multicast actors into a graph — the streams exist, but nothing yet says *where
they meet*. That meeting place is the next, distinctly-multicast idea.


## Step 2 — The Rendezvous Point: the meeting place unicast never had

**Theory.** Here is what makes sparse-mode *sparse*. Unlike dense-mode's flood-everywhere, a
sparse-mode group builds a **shared tree** rooted at a single agreed router: the **Rendezvous
Point (RP)**. Sources *register* to the RP; receivers *join toward* the RP. Until a group has a
reachable RP, its shared tree cannot form at all — no RP, no tree, no traffic.

That is a dependency unicast simply does not have. A unicast destination doesn't need a third
router to introduce it to its listeners. A sparse-mode group *does*. So the RP is a **single
point** sitting quietly in the middle of your design — and if it fails, every group mapped to
it goes dark at once, while unicast reachability to that same RP box looks perfectly normal.

*(One honest nuance a multicast engineer will hold me to: in practice, an already-flowing
source tree — after SPT-switchover — can survive an RP outage, because it's built RPF-toward
the source, not toward the RP; the RP mainly gates new joins and new rendezvous. We model the
conservative case — a group still dependent on shared-tree join-through-RP — because that's the
risk you're designing against.)*

In the graph, each RP becomes a node with an `RP` **label**, an `address`, and — the fact that
matters most — a `state` of `up` or `down`. Each group points at its RP with a `ROOTED_AT`
**edge**. We model two RPs, and we bake the incident in from the start:

- **`rp-core-1`** roots the IPTV group `239.1.1.1` — and it is **down**. This is our 3 AM event.
- **`rp-core-2`** roots the market-data group `239.2.2.2` — and it is **up** and healthy.

One RP down on purpose — so the failure, and its blast radius, shows up cleanly on one node you
can point at.


In [ ]:
G.add_node('rp-core-1', label='RP', address='10.0.0.1', state='down')   # <-- the incident
G.add_node('rp-core-2', label='RP', address='10.0.0.2', state='up')

# ROOTED_AT: a group's shared tree meets at its RP.
G.add_edge('239.1.1.1', 'rp-core-1', rel='ROOTED_AT')
G.add_edge('239.2.2.2', 'rp-core-2', rel='ROOTED_AT')

for n, d in G.nodes(data=True):
    if d.get('label') == 'RP':
        print(f"{n}: address={d['address']}, state={d['state']}")

**Checkpoint.** Two RP nodes print, and exactly one reads `state=down`: `rp-core-1`. The IPTV
group is rooted at that dead RP; the market-data group is rooted at a live one. The whole
incident now sits in your graph as a single fact on a single node — and every group that
depends on it is one `ROOTED_AT` edge away.


## Step 3 — The distribution tree: the branch carries the state (this is the pivotal step)

**Theory.** The RP is the *root* of the tree; the tree itself is the chain of routers that
forward the group downward, hop by hop, toward the receivers. Each hop is a **branch**. And
here is the idea the whole lab pivots on: when a branch stops forwarding a group, *where* does
that fact belong? Not on the router above it (it's still forwarding other groups). Not on the
router below it (it's fine). It belongs on the **branch between them** — the forwarding
relationship. PIM already agrees: an interface either has the group in its outgoing list, or it
doesn't.

So we add **edges** with a `rel` of `FORWARDS`, and we hang two properties on each: the `group`
it carries and its `state`. Read `dist-1 --FORWARDS(239.2.2.2, up)--> lhr-md` literally:
*"dist-1 forwards the market-data group down to lhr-md, and that branch is up."* We build a
two-hop tree under each RP, down to a last-hop router (`lhr-`) where receivers attach. Every
branch starts **up** — we'll flip one later and watch a group go dark from a *branch* failure,
the same idea as the RP failure but with the fact living on an edge instead of a node.


In [ ]:
# Tree nodes: aggregation + last-hop routers where receivers attach.
for r in ['dist-iptv', 'lhr-iptv', 'dist-md', 'lhr-md']:
    G.add_node(r, label='Router')

# add_edge(source, target, **properties). The branch state is a property ON the edge.
G.add_edge('rp-core-1', 'dist-iptv', rel='FORWARDS', group='239.1.1.1', state='up')
G.add_edge('dist-iptv', 'lhr-iptv',  rel='FORWARDS', group='239.1.1.1', state='up')
G.add_edge('rp-core-2', 'dist-md',   rel='FORWARDS', group='239.2.2.2', state='up')
G.add_edge('dist-md',   'lhr-md',    rel='FORWARDS', group='239.2.2.2', state='up')

for u, v, d in G.edges(data=True):
    if d.get('rel') == 'FORWARDS':
        print(f"{u} --FORWARDS({d['group']}, {d['state']})--> {v}")

**Checkpoint.** Four `FORWARDS` branches print, each tagged with the group it carries and a
`state` of `up`. Two branches build the IPTV tree under `rp-core-1`, two build the market-data
tree under `rp-core-2`. The tree is now a set of facts-on-edges you can query against — even
though, for IPTV, the root above them is already dead.


## See it — draw the graph

**Theory.** Text is great during an incident, but to *understand* a design you want the
picture. We'll colour nodes by their **label** (sources, groups, RPs, routers each a colour),
outline any **down** node in red, and draw any **down** branch as a dashed red arrow. This is
the same instinct as a topology diagram — except this diagram is generated *from the data*, so
it can never drift out of sync with the truth.


In [ ]:
def draw(G, title='Multicast (PIM-SM) knowledge graph'):
    colors = {'Source': '#4c9a6a', 'Group': '#0f7f74', 'RP': '#9b59b6',
              'Router': '#3aa0ff', 'Service': '#c8702a'}
    node_colors = [colors.get(G.nodes[n].get('label'), '#cccccc') for n in G]
    # red outline on any node whose state is down (a dead RP jumps out)
    borders = ['#d34b3f' if G.nodes[n].get('state') == 'down' else '#2b3640' for n in G]
    widths  = [2.6 if G.nodes[n].get('state') == 'down' else 0.8 for n in G]
    pos = nx.spring_layout(G, seed=7)   # fixed seed => stable, repeatable layout

    plt.figure(figsize=(11, 7.5))
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1900,
                           alpha=0.92, edgecolors=borders, linewidths=widths)
    nx.draw_networkx_labels(G, pos, font_size=8)

    down  = [(u, v) for u, v, d in G.edges(data=True) if d.get('state') == 'down']
    other = [(u, v) for u, v, d in G.edges(data=True) if (u, v) not in down]
    # curve edges slightly so any reciprocal pair never draws on top of itself
    nx.draw_networkx_edges(G, pos, edgelist=other, arrows=True, arrowsize=15,
                           edge_color='#8894a0', connectionstyle='arc3,rad=0.12')
    nx.draw_networkx_edges(G, pos, edgelist=down, arrows=True, arrowsize=15, width=2.0,
                           edge_color='#d34b3f', style='dashed', connectionstyle='arc3,rad=0.12')
    nx.draw_networkx_edge_labels(G, pos, font_size=6.5, label_pos=0.5,
        edge_labels={(u, v): d.get('rel', '') for u, v, d in G.edges(data=True)})

    plt.axis('off'); plt.title(title); plt.tight_layout(); plt.show()

draw(G)

**Reading the picture.** You should see the sources (green) feeding the groups (teal), each
group `ROOTED_AT` an RP (purple), and the routers (blue) fanning the trees downward. One RP,
`rp-core-1`, is outlined in **red** — the down RP rooting the IPTV group. This is still just a
topology. It becomes a *knowledge* graph in the next step, when we add the thing a topology
diagram has never carried: the business behind the stream.


## Step 4 — The business service: the node your routers never had

**Theory.** This is the node PIM was never designed to hold, and the reason the whole lab
exists. Your routers know sources, groups, RPs, trees, RPF checks. They have never once known
that `239.1.1.1` *is* the **IPTV Channel Lineup** every trading-floor screen is watching, and
that when that group loses its tree, **a room full of people lose their feed**.

So we add a **Service** node for each stream, with a `criticality` property, and wire it to its
group with a `CONSUMES` edge: *"this service consumes that group."* One edge — and now a
multicast fact and a business fact are welded together in a place you can query:

- **`IPTV Channel Lineup`** consumes `239.1.1.1`.
- **`Market Data Feed`** consumes `239.2.2.2`.

No `show` command holds these links. They have never lived anywhere except tribal knowledge.
You're about to make them first-class, walkable facts.


In [ ]:
G.add_node('IPTV Channel Lineup', label='Service', criticality='critical')
G.add_node('Market Data Feed',    label='Service', criticality='critical')

G.add_edge('IPTV Channel Lineup', '239.1.1.1', rel='CONSUMES')
G.add_edge('Market Data Feed',    '239.2.2.2', rel='CONSUMES')

print('Graph now has', G.number_of_nodes(), 'nodes and', G.number_of_edges(), 'edges.')
print('The load-bearing links:', [f'{u} -CONSUMES-> {v}'
      for u, v, d in G.edges(data=True) if d.get('rel') == 'CONSUMES'])
draw(G, title='Multicast knowledge graph — now with the business services')

**Checkpoint.** The graph has grown two orange `Service` nodes, each joined by a `CONSUMES`
edge to its group. In the redrawn picture you can *trace with your finger*: dead RP →
`239.1.1.1` → `IPTV Channel Lineup`. In the next step we make the computer trace it for you.


## Step 5 — The 3 AM question: blast radius as a traversal

**Theory.** Everything so far was setup for this. A **traversal** is just walking edges to
answer a question. Ours answers:

> *"Which business services ride a group whose shared tree is broken right now?"*

And a group's tree is broken two ways — the two distinctly-multicast failure modes:

1. **RP failure** — *every* RP the group is `ROOTED_AT` is `down`. With no reachable RP, the
   shared tree can't form. (One RP today; we'll add a backup later — that's the anycast lesson.)
2. **Branch failure** — a `FORWARDS` branch carrying that group is `down`, cutting off everything
   downstream of it.

We write a small helper, `group_is_dark`, that checks both, then walk every `CONSUMES` edge to
surface the services on any dark group. Crucially, the query is **conditioned on state** — flip
the RP back up (or the branch), and the same walk returns nothing. That responsiveness is the
difference between a static diagram and a live model. Run it.


In [ ]:
def group_is_dark(G, group):
    """A group's shared tree is broken if it has no reachable RP, or a branch is down."""
    # 1) RP failure: every RP this group is ROOTED_AT is down
    rps = [rp for _, rp, d in G.out_edges(group, data=True) if d.get('rel') == 'ROOTED_AT']
    if rps and not any(G.nodes[rp].get('state') == 'up' for rp in rps):
        down = [rp for rp in rps if G.nodes[rp].get('state') == 'down']
        return True, 'no reachable RP (' + ', '.join(down) + ' down)'
    # 2) branch failure: a FORWARDS branch carrying this group is down
    for u, v, d in G.edges(data=True):
        if d.get('rel') == 'FORWARDS' and d.get('group') == group and d.get('state') == 'down':
            return True, f'branch down ({u} -> {v})'
    return False, ''

def blast_radius(G):
    """Services put at risk by any multicast group whose tree is currently broken."""
    at_risk = []
    for svc, grp, d in G.edges(data=True):
        if d.get('rel') != 'CONSUMES':
            continue
        dark, reason = group_is_dark(G, grp)
        if dark:
            at_risk.append((svc, grp, reason))
    return at_risk

hits = blast_radius(G)
for svc, grp, reason in hits:
    print(f'{grp} dark  ->  {reason}  ->  AT RISK: {svc}')
if not hits:
    print('nothing at risk')

The router only ever told you an RP process died — one line in a log. Your graph just told you
the **IPTV Channel Lineup is at risk**, and showed its work: the dead RP, the group rooted at
it, and the service riding that group. Market Data, rooted at a healthy RP, correctly stays
silent. You got that answer because *you* added the one node PIM never had. That is the entire
pitch of a knowledge graph, and you just built it from an empty page.


## Step 6 — What-if: repair the RP, break a branch, put it all back

**Theory.** Because the failures live as plain properties — a `state` on the RP node, a `state`
on the branch edge — you can *change* them and re-ask, running "what if this fails?" (or "what
if I fix it?") on a **model**, safely, with no one paged and no maintenance window. Watch two
distinct multicast failures move the answer:

1. Bring `rp-core-1` back **up** — the IPTV blast radius clears.
2. Now cut a **market-data branch** (`dist-md -> lhr-md`) — a *different* service, `Market Data
   Feed`, goes dark, this time from a branch failure rather than an RP failure.
3. Put both back — the graph returns to all-clear.

Same graph, same query — only the state on one node or one edge changes each time.


In [ ]:
G.nodes['rp-core-1']['state'] = 'up'          # 1) repair the RP
print('After RP repair:      ', sorted({s for s, *_ in blast_radius(G)}) or 'nothing at risk')

G['dist-md']['lhr-md']['state'] = 'down'      # 2) cut a market-data branch
print('After branch cut:     ', [f'{s} ({r})' for s, g, r in blast_radius(G)])

G.nodes['rp-core-1']['state'] = 'down'        # 3a) put the RP failure back
G['dist-md']['lhr-md']['state'] = 'up'        # 3b) heal the branch
print('Back to the incident: ', sorted({s for s, *_ in blast_radius(G)}))

**Checkpoint.** After the RP repair you see `nothing at risk`. Cut the market-data branch and
`Market Data Feed` appears — flagged `branch down (dist-md -> lhr-md)`, a different failure mode
than the RP one. Restore both and you're back to `['IPTV Channel Lineup']`, the original
incident. The answer *responded* to each change. That is what makes it a model rather than a
drawing.


## Your turn #1 — a second service on the same group

Real multicast groups rarely feed just one consumer. Suppose a **Trading Floor Video Wall**
also subscribes to the IPTV group `239.1.1.1`. Add it, wire it with a `CONSUMES` edge, and
re-run `blast_radius`. You should now see **two** services surface from the same down RP —
because one edge of extra truth is all it takes.

Fill in the cell below (there's a `# TODO`), then run it. The solution follows if you want to
check.


In [ ]:
# TODO: add a 'Trading Floor Video Wall' Service node (criticality='high'),
#       then a CONSUMES edge from it to the group '239.1.1.1'.

# G.add_node(...)
# G.add_edge(...)

for svc, grp, reason in blast_radius(G):
    print(f'AT RISK: {svc}  (via {grp}: {reason})')

<details><summary><b>Solution — click to reveal</b></summary>

```python
G.add_node('Trading Floor Video Wall', label='Service', criticality='high')
G.add_edge('Trading Floor Video Wall', '239.1.1.1', rel='CONSUMES')

for svc, grp, reason in blast_radius(G):
    print(f'AT RISK: {svc}  (via {grp}: {reason})')
```

Now both `IPTV Channel Lineup` **and** `Trading Floor Video Wall` come back from the one dead
RP. The blast radius grew the instant you told the graph one more true thing — you didn't touch
the query at all.
</details>


In [ ]:
# (Solution, applied — so the rest of the notebook has both IPTV services in the graph.)
G.add_node('Trading Floor Video Wall', label='Service', criticality='high')
G.add_edge('Trading Floor Video Wall', '239.1.1.1', rel='CONSUMES')
print('Services at risk now:', sorted({s for s, *_ in blast_radius(G)}))

## Quiet reveal — you have been writing an *ontology*

Time to name the thing you've been doing. Every step, you made two very different kinds of
decision, and it's worth seeing the seam between them:

- *"A `Group` is `ROOTED_AT` an `RP`. A `Source` `SENDS_TO` a `Group`. A `Service` `CONSUMES` a
  `Group`. A `Router` `FORWARDS` a group to another `Router`."* — these are the **rules**: which
  node types exist, which edges are allowed, what shape a valid fact takes. That is the
  **schema**. Its fancy name is an **ontology** — and here's the friendly definition: *an
  ontology is the RFC of your graph, the agreed vocabulary written down as structure.* You
  already trust RFC 7761 to say what a valid PIM-SM adjacency is; an ontology does the same job
  for your graph.

- *"This particular RP is `rp-core-1`, address `10.0.0.1`, and its state is `down`."* — these
  are the **instances**: the actual data.

A rule of thumb that keeps the two straight forever: **if it has an address, a hostname, or a
group number, it is data (an instance), never schema.** `RP` is schema; `rp-core-1` is data.
`CONSUMES` is schema; "IPTV Channel Lineup consumes 239.1.1.1" is data. Keep the vocabulary
small and stable; let the instances be many. That single discipline is the difference between a
graph you can grow for years and a swamp you abandon in a month.


## A peek at the real thing (1/2) — the same RP and group in Neo4j + Cypher

We used networkx so you could run everything with zero setup. But the *shapes* you built are
exactly what you'd build in a real graph database like **Neo4j**, whose query language is
**Cypher**. Cypher is worth a glance because it reads almost like the arrows we've been drawing
— `(node)-[:EDGE]->(node)`.

Here is **Step 2 (the RP and the group rooted at it)** as Cypher. `MERGE` means "find this node
or create it if missing" (so re-running is safe); `SET` assigns properties. This is *reference
only* — you don't run it here, it just shows you the same idea in the grown-up tool:

```cypher
MERGE (rp:RendezvousPoint {id: 'rp-core-1'})
SET   rp.address = '10.0.0.1', rp.state = 'down';

MERGE (g:MulticastGroup {id: '239.1.1.1'})
SET   g.group_address = '239.1.1.1', g.app = 'IPTV';

// the group's shared tree meets at its RP — same as our ROOTED_AT edge
MERGE (g)-[r:ROOTED_AT]->(rp)
SET   r.tree = 'shared';
```

See it? `(g)-[:ROOTED_AT]->(rp)` is the same statement as our
`G.add_edge('239.1.1.1', 'rp-core-1', rel='ROOTED_AT')`. Same nodes, same edge, same
dependency — one just happens to live in a database that scales to millions of nodes. Note the
idempotent `MERGE ... SET` shape: run it twice and you get one RP, not two.


## A peek at the real thing (2/2) — the 3 AM question in Cypher

Our blast-radius walk is a Python helper plus a loop. In Cypher, the RP-failure half of that
traversal is a few lines, because in a graph database you **draw the shape you're looking for**
and let the engine find every match — no manual loops:

```cypher
MATCH (svc:BusinessService)-[:CONSUMES]->(g:MulticastGroup)-[:ROOTED_AT]->(rp:RendezvousPoint)
WITH  svc, g, collect(rp) AS rps
WHERE none(rp IN rps WHERE rp.state = 'up')   // no reachable RP for this group
RETURN g.group_address AS group,
       collect(DISTINCT svc.name) AS services_at_risk;
```

Read the `WHERE` line as a sentence: *"keep only the groups where none of their RPs is up."*
That's the same RP check you wrote in Python — the pattern you `MATCH` **is** the traversal. Run
it against the real dataset and `239.1.1.1` comes back with `IPTV Channel Lineup` among the
services at risk. Our tiny in-memory graph models just the streams we need to keep the story
clean. Different engine; identical thinking. If you can read the Python above, you can read the
Cypher — you already speak this language.


## Your turn #2 — anycast-RP redundancy: remove the single point

Here is the fix a multicast designer reaches for, and it maps perfectly onto the graph. The
reason the IPTV group dies is that it has **exactly one** RP, and that RP is down. **Anycast-RP**
removes the single point: two (or more) routers share the *same* RP address, kept in sync by
MSDP, so if one dies the other still roots the tree. In our model that is simply **a second
`ROOTED_AT` edge to a healthy RP** — and because `group_is_dark` already treats a group as dark
only when *every* RP is down, the redundancy just works.

1. add a backup RP node `rp-core-3` (`state='up'`) — the anycast partner;
2. add a `ROOTED_AT` edge from `239.1.1.1` to `rp-core-3`;
3. re-run `blast_radius` — the IPTV services **clear**, even though `rp-core-1` is still down.

Fill in the `# TODO`s, then run. The solution follows.


In [ ]:
# TODO 1: add RP 'rp-core-3' (label='RP', address='10.0.0.1', state='up')  # anycast: shares the RP address
# TODO 2: add a ROOTED_AT edge from '239.1.1.1' to 'rp-core-3'

# G.add_node(...)
# G.add_edge(...)

print('IPTV group rooted at:', [rp for _, rp, d in G.out_edges('239.1.1.1', data=True)
                                if d.get('rel') == 'ROOTED_AT'])
print('Services at risk now:', sorted({s for s, *_ in blast_radius(G)}) or 'nothing at risk')

<details><summary><b>Solution — click to reveal</b></summary>

```python
G.add_node('rp-core-3', label='RP', address='10.0.0.1', state='up')  # anycast partner of rp-core-1
G.add_edge('239.1.1.1', 'rp-core-3', rel='ROOTED_AT')

print('IPTV group rooted at:', [rp for _, rp, d in G.out_edges('239.1.1.1', data=True)
                                if d.get('rel') == 'ROOTED_AT'])
print('Services at risk now:', sorted({s for s, *_ in blast_radius(G)}) or 'nothing at risk')
```

Before, the IPTV group had a single RP and that RP was down, so its services were at risk. The
moment you gave it a second, healthy RP at the same anycast address, `group_is_dark` saw *a*
reachable RP and stopped flagging it — the services cleared, with `rp-core-1` still dead. You
didn't change the query; you changed the design, and the traversal told you the single point
was gone. That is exactly why anycast-RP exists.
</details>


In [ ]:
# (Solution, applied — then we remove the backup again so later cells start from the incident.)
G.add_node('rp-core-3', label='RP', address='10.0.0.1', state='up')
G.add_edge('239.1.1.1', 'rp-core-3', rel='ROOTED_AT')
print('With anycast backup:  ', sorted({s for s, *_ in blast_radius(G)}) or 'nothing at risk')

G.remove_node('rp-core-3')   # restore the single-RP incident for the rest of the lab
print('Backup removed again: ', sorted({s for s, *_ in blast_radius(G)}))

## Make it yours — swap in *your* multicast

Now the moment it becomes your tool, not mine. Change the four values below to **one** group,
**one** RP, **one** source, and **one** service *you* actually run. We add the group `ROOTED_AT`
a **down** RP for you, so your service shows up. Run it, and watch **your own service name**
come back from `blast_radius` — proof that the machine you built understands your network, not
just the demo's.

Keep it to the smallest slice that answers one real question. You can always add one more node
tomorrow — that's the whole promise of a graph you can grow.


In [ ]:
# --- change these four values to your network ---
MY_GROUP   = '239.9.9.9'
MY_RP      = 'rp-campus-1'
MY_SOURCE  = 'imaging-modality'
MY_SERVICE = 'Radiology Imaging'
# ------------------------------------------------

G.add_node(MY_GROUP,   label='Group',   group_address=MY_GROUP, app='custom')
G.add_node(MY_RP,      label='RP',       address='10.0.0.9', state='down')   # your modelled failure
G.add_node(MY_SOURCE,  label='Source',   address='10.30.0.5')
G.add_node(MY_SERVICE, label='Service',  criticality='critical')

G.add_edge(MY_SOURCE,  MY_GROUP,   rel='SENDS_TO')
G.add_edge(MY_GROUP,   MY_RP,      rel='ROOTED_AT')
G.add_edge(MY_SERVICE, MY_GROUP,   rel='CONSUMES')

print(f'If {MY_RP} is down, these services are at risk:')
for svc, grp, reason in blast_radius(G):
    if grp == MY_GROUP:
        print(f'  AT RISK: {svc}   (via {grp}: {reason})')

**Checkpoint.** Your own service name — `Radiology Imaging`, or whatever you typed — prints as
*at risk*, with the reason naming your down RP. That is the moment the tool stopped being a
tutorial and became yours. Every other stream you run is just four more lines away.


## The one rule that keeps this from becoming a swamp

You'll be tempted to model everything. Don't. Here is the discipline, in one line:

> **Model the nouns of your multicast design review, not the packets on the wire.**

Sources, groups, RPs, trees, branches, boundaries, services — the **nouns** you'd draw on a
whiteboard while arguing about a multicast design — those belong in the graph. Per-flow packet
rates, IGMP join/leave timers, the full (S,G) mroute table, SPT-switchover counters — those are
the **numbers**; leave them in the systems that already store them well, and let the graph
*reference* them when it needs to. The graph holds the *shape of the dependency*; your NMS
holds the firehose. Keep that line sharp and your graph stays queryable for years.


## You built a knowledge graph

Look back at the distance. You started with an empty page and five plain words. You added
sources, groups, RPs, and a distribution tree — a topology. Then you added the one node PIM
never had, the **business service** — and the topology became *knowledge*. Finally you asked it
the question no router can answer, and it printed **IPTV Channel Lineup**, then responded
correctly every time you changed the world underneath it — a dead RP, a cut branch, an anycast
backup that made the single point disappear.

The important idea was never multicast, and never networkx. It's this: **operational truth has
a shape** — sources feed groups, groups root at RPs, trees branch down to receivers, services
depend on groups — and once that shape is a graph, impact analysis stops being tribal knowledge
and becomes a traversal.

### Where to go next
- **The companion Neo4j lab** does this exact thing in a real graph database — same nodes, same
  edges, same 3 AM question — so the two Cypher peeks above become things you run.
- **Add RPF.** Reverse-path forwarding is the other multicast gotcha: model the unicast route
  each RPF check leans on, and "receiver joined but traffic never arrives" becomes one traversal.
- **Add the change layer.** Model a `ChangeEvent` node linked to what it touched, and "what
  changed right before this group went dark?" becomes one more traversal.

You already run a distribution tree in every corner of your network. Now you know how to build
the graph that holds the part `show ip mroute` was never asked to remember. Go model one real
stream on your network, and ask it what breaks.
